# 38 -- Section 01e Step 2: break x into the Rumble term and the other 529 movies

Reuses notebook 37's exact user, movie, seeds and starting weights. Splits
x+ and x- (01c scenario) into Rumble's own term and the sum over the other
529 rated movies, and re-derives the 01e values via b times the weight sum
over the whole rated movie list. Asserts every identity the HTML will
state before any number is written into the page. Does not touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import torch

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "rumble_term_breakdown" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step2_rumble_term_breakdown.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  37  (37_step1to6_rating_habit_b.ipynb)
this notebook is:           38  (38_step2_rumble_term_breakdown.ipynb)


## Part 1 -- Reuse notebook 37's exact setup

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
results37_path = out_dir / "37_step1to6_rating_habit_b_results.json"
for p in (train_path, test_path, results31_path, results37_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474

results31 = json.loads(results31_path.read_text())
results37 = json.loads(results37_path.read_text())
REAL_SEED = results31["seeds"]["plus_seed_reused"]
MINUS_SEED = results31["seeds"]["minus_seed_trained"]
N_HIDDEN = int(results31["architecture_match"]["128 hidden units"].split("=")[1])
H1_IDX = 0

all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_movies = len(all_movies)
movie_to_col = {m: j for j, m in enumerate(all_movies)}

movies = pd.read_csv(data_dir / "movie.csv")
RUMBLE_ID = int(movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)].iloc[0]["movieId"])
rumble_col = movie_to_col[RUMBLE_ID]
MEDIAN_USER = results31["worked_example_plus"]["user"]

user_rows = train_hyp[train_hyp["userId"] == MEDIAN_USER]
v0_full = torch.zeros(n_movies)
mask0_full = torch.zeros(n_movies)
for row in user_rows.itertuples():
    v0_full[movie_to_col[row.movieId]] = row.a
    mask0_full[movie_to_col[row.movieId]] = 1.0
rated_idx = mask0_full.nonzero(as_tuple=True)[0]
a_rumble = v0_full[rumble_col].item()
n_rated = len(rated_idx)
assert n_rated == results37["step2"]["R_size"] == 530

g_init_p = torch.Generator().manual_seed(REAL_SEED)
W_plus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_p) * 0.01
g_init_m = torch.Generator().manual_seed(MINUS_SEED)
W_minus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_m) * 0.01

b_val = results37["step1"]["b"]
w_plus_rumble = W_plus_init[rumble_col, H1_IDX].item()
w_minus_rumble = W_minus_init[rumble_col, H1_IDX].item()
print(f"a={a_rumble:.6f}  b={b_val:.6f}  n_rated={n_rated}")
print(f"w+[Rumble,h1]={w_plus_rumble:.6f}  w-[Rumble,h1]={w_minus_rumble:.6f}")

a=0.777778  b=0.656813  n_rated=530
w+[Rumble,h1]=0.006433  w-[Rumble,h1]=-0.009362


## Part 2 -- Rumble term, other-529 sum, and the two weight sums over R

In [3]:
x_plus_01c = (v0_full @ W_plus_init)[H1_IDX].item()
x_minus_01c = (v0_full @ W_minus_init)[H1_IDX].item()
assert abs(x_plus_01c - results37["step2"]["x_plus"]["01c"]) < 1e-6
assert abs(x_minus_01c - results37["step2"]["x_minus"]["01c"]) < 1e-6

rumble_term_plus = a_rumble * w_plus_rumble
rumble_term_minus = a_rumble * w_minus_rumble
rest_plus = x_plus_01c - rumble_term_plus
rest_minus = x_minus_01c - rumble_term_minus

assert abs((rumble_term_plus + rest_plus) - x_plus_01c) < 1e-6
assert abs((rumble_term_minus + rest_minus) - x_minus_01c) < 1e-6
print(f"Rumble term (+): {rumble_term_plus:.6f}   rest (other {n_rated-1} movies): {rest_plus:.6f}   sum: {rumble_term_plus+rest_plus:.6f}")
print(f"Rumble term (-): {rumble_term_minus:.6f}   rest (other {n_rated-1} movies): {rest_minus:.6f}   sum: {rumble_term_minus+rest_minus:.6f}")

sum_w_plus_R = W_plus_init[rated_idx, H1_IDX].sum().item()
sum_w_minus_R = W_minus_init[rated_idx, H1_IDX].sum().item()
assert abs(sum_w_plus_R - results37["step2"]["sum_w_plus_R"]) < 1e-6
assert abs(sum_w_minus_R - results37["step2"]["sum_w_minus_R"]) < 1e-6
print(f"sum w+ over R = {sum_w_plus_R:.6f}   sum w- over R = {sum_w_minus_R:.6f}")

Rumble term (+): 0.005004   rest (other 529 movies): -0.216119   sum: -0.211115
Rumble term (-): -0.007281   rest (other 529 movies): -0.198086   sum: -0.205367
sum w+ over R = -0.332340   sum w- over R = -0.325376


## Part 3 -- 01e values and the two products, cross-checked against notebook 37

In [4]:
product_plus = b_val * sum_w_plus_R
product_minus = b_val * sum_w_minus_R
x_plus_01e = x_plus_01c + product_plus
x_minus_01e = x_minus_01c - product_minus

assert abs(x_plus_01e - results37["step2"]["x_plus"]["01e"]) < 1e-6
assert abs(x_minus_01e - results37["step2"]["x_minus"]["01e"]) < 1e-6

channel_plus_rumble_01c = a_rumble
channel_minus_rumble_01c = a_rumble
channel_plus_rumble_01e = a_rumble + b_val
channel_minus_rumble_01e = a_rumble - b_val
assert abs(channel_plus_rumble_01e - results37["step1"]["channel_plus_rumble"]["01e"]) < 1e-6
assert abs(channel_minus_rumble_01e - results37["step1"]["channel_minus_rumble"]["01e"]) < 1e-6

print(f"product (+) = b * sum_w_plus_R = {product_plus:.6f}")
print(f"product (-) = b * sum_w_minus_R = {product_minus:.6f}")
print(f"x+_01e = {x_plus_01e:.6f}  (matches notebook 37 and the current table)")
print(f"x-_01e = {x_minus_01e:.6f}  (matches notebook 37 and the current table)")
print("All identities confirmed to 1e-6. OK.")

product (+) = b * sum_w_plus_R = -0.218285
product (-) = b * sum_w_minus_R = -0.213712
x+_01e = -0.429400  (matches notebook 37 and the current table)
x-_01e = 0.008345  (matches notebook 37 and the current table)
All identities confirmed to 1e-6. OK.


## Part 4 -- Save for the rebuilt Step 2 worked example

In [5]:
results = {
    "notebook": this_name,
    "reused_from": "31_step8_minus_channel.ipynb, 37_step1to6_rating_habit_b.ipynb",
    "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
    "user": MEDIAN_USER,
    "hidden_unit": "h1 (index 0)",
    "a": a_rumble,
    "b": b_val,
    "n_rated": n_rated,
    "channel_plus_rumble": {"01c": channel_plus_rumble_01c, "01e": channel_plus_rumble_01e},
    "channel_minus_rumble": {"01c": channel_minus_rumble_01c, "01e": channel_minus_rumble_01e},
    "rumble_term_plus": rumble_term_plus,
    "rumble_term_minus": rumble_term_minus,
    "rest_plus": rest_plus,
    "rest_minus": rest_minus,
    "x_plus": {"01c": x_plus_01c, "01e": x_plus_01e},
    "x_minus": {"01c": x_minus_01c, "01e": x_minus_01e},
    "sum_w_plus_R": sum_w_plus_R,
    "sum_w_minus_R": sum_w_minus_R,
    "product_plus": product_plus,
    "product_minus": product_minus,
}
results_path = out_dir / "38_step2_rumble_term_breakdown_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps(results, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/38_step2_rumble_term_breakdown_results.json
{
  "notebook": "38_step2_rumble_term_breakdown.ipynb",
  "reused_from": "31_step8_minus_channel.ipynb, 37_step1to6_rating_habit_b.ipynb",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "user": 24127,
  "hidden_unit": "h1 (index 0)",
  "a": 0.7777777910232544,
  "b": 0.6568134171907757,
  "n_rated": 530,
  "channel_plus_rumble": {
    "01c": 0.7777777910232544,
    "01e": 1.4345912082140302
  },
  "channel_minus_rumble": {
    "01c": 0.7777777910232544,
    "01e": 0.12096437383247871
  },
  "rumble_term_plus": 0.005003613556781972,
  "rumble_term_minus": -0.007281473133953331,
  "rest_plus": -0.21611873539821264,
  "rest_minus": -0.1980855406781118,
  "x_plus": {
    "01c": -0.21111512184143066,
    "01e": -0.4294003180925201
  },
  "x_minus": {
    "01c": -0.20536701381206512,
    "01e": 0.008344604852814352
  },
  "sum_w_plus_R": -0.33233973383903503,
  "sum_